# 1 — Create Lease (Jellyfin Recommender Serving)

This notebook reserves a **GPU bare-metal node** on Chameleon (`CHI@UC`, `gpu_rtx_6000`) that will be used to run all five serving experiments.

**Before you start:**
- Replace `YOUR_NET_ID` with your actual NYU Net ID.
- Make sure you are logged in to the Chameleon Jupyter environment.

In [ ]:
# runs in Chameleon Jupyter environment
from chi import server, context, lease
import os

context.version = "1.0"
context.choose_project()
context.choose_site(default="CHI@UC")

In [ ]:
# runs in Chameleon Jupyter environment
# CHANGE THIS to your Net ID
NET_ID = "YOUR_NET_ID"

LEASE_NAME = f"serve_jellyfin_{NET_ID}"
NODE_TYPE  = "gpu_rtx_6000"   # NVIDIA RTX 6000 — single-GPU bare metal at CHI@UC

print(f"Lease name : {LEASE_NAME}")
print(f"Node type  : {NODE_TYPE}")

In [ ]:
# runs in Chameleon Jupyter environment
# Reserve the node for 8 hours (adjust start/end if needed)
# Book at least ONE DAY in advance for gpu_rtx_6000
import datetime

start_time = datetime.datetime.now(datetime.timezone.utc) + datetime.timedelta(minutes=5)
end_time   = start_time + datetime.timedelta(hours=8)

l = lease.Lease(
    name=LEASE_NAME,
    start_date=start_time,
    end_date=end_time,
)
l.add_node_reservation(count=1, node_type=NODE_TYPE)
l.submit(idempotent=True)
l.show()

In [ ]:
# runs in Chameleon Jupyter environment
# Poll until the lease is ACTIVE
import time

while True:
    l = lease.get_lease(LEASE_NAME)
    status = l.status
    print(f"Lease status: {status}")
    if status == "ACTIVE":
        break
    elif status in ("ERROR", "DELETED"):
        raise RuntimeError(f"Lease ended up in status: {status}")
    time.sleep(30)

print("Lease is ACTIVE — proceed to notebook 2.")
l.show()